# 04 — Cải tiến & Train Model v2

## Vì sao baseline yếu trên camera LIVE?

| Hiện tượng bạn gặp | Nguyên nhân khả dĩ | Hướng xử lý v2 |
|--------------------|--------------------|----------------|
| Chỉ nhận khi tờ che ~70% khung hình | Train/val nhiều ảnh “studio”, object lớn; webcam nền rối (mặt/tường/chăn) lệch phân phối | Augment **scale mạnh** + mosaic/copy-paste; (nên) thêm ảnh nền rối |
| Khó nhận 2 tờ cùng lúc | NMS/conf + thiếu ngữ cảnh nhiều object khó; mosaic mặc định chưa đủ “khó” | mosaic + mixup + copy_paste; conf demo thấp hơn một chút |
| 1000/2000/5000 ↔ 20000/50000 | Màu/họa tiết gần nhau trên webcam nén/nhiễu | Model **YOLOv8s** + tăng **cls loss** + HSV/ánh sáng |
| 5000 mặt sau gần như không ra | Dữ liệu train lệch **mặt trước**; mặt sau là class nội tại khác về texture | **Bổ sung ảnh mặt sau** (bắt buộc nếu muốn ổn định) |

> Val mAP baseline ~0.99 **không mâu thuẫn** với demo xấu: metric trên split Roboflow dễ hơn camera thật (domain gap).

**Notebook này:** chốt recipe v2 → (tuỳ chọn) chạy train tại chỗ → so với baseline.  
**Khuyến nghị GPU mạnh:** dùng `train_v2.py` trên server (giống lần baseline).

## 0. Import & đường dẫn

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from ultralytics import YOLO

sns.set_theme(style="whitegrid")

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from train_baseline import build_clean_dataset, write_data_yaml, CLEAN_YAML, CLEAN_DATA_DIR

BASELINE_PT = NOTEBOOK_DIR / "baseline_exports" / "yolov8n_baseline_best.pt"
BASELINE_METRICS = NOTEBOOK_DIR / "baseline_exports" / "baseline_val_metrics.csv"
V2_EXPORT = NOTEBOOK_DIR / "v2_exports"
V2_EXPORT.mkdir(parents=True, exist_ok=True)
RUNS_DIR = PROJECT_DIR / "runs" / "detect"

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("CLEAN_YAML:", CLEAN_YAML)
print("Baseline weights exist:", BASELINE_PT.exists())

## 1. Checklist dữ liệu nên bổ sung (ưu tiên cao hơn chỉ tăng epoch)

Trước/ song song với train v2, nhóm nên **tự chụp thêm** (gán nhãn Roboflow rồi merge):

1. **Nền rối:** mặt người, tường, giường/chăn, bàn làm việc — tờ tiền chỉ chiếm 15–40% khung hình  
2. **Nhiều tờ:** 2–4 tờ cạnh nhau / hơi chồng  
3. **Cặp dễ nhầm:** 1000 vs 20000, 2000 vs 20000, 5000 vs 20000/50000 — cùng điều kiện ánh sáng  
4. **Mặt sau:** đặc biệt `5000`, thêm `1000`, `2000`  
5. **Webcam / điện thoại:** cùng loại cam sẽ dùng khi demo

Nếu chưa kịp bổ sung dữ liệu: vẫn train v2 bằng augment mạnh (dưới đây) — giúp một phần, **không thay** được ảnh mặt sau thật.

In [ ]:
# Đảm bảo dataset bbox sạch sẵn sàng
build_clean_dataset(force=False)
CLASS_NAMES = write_data_yaml()
print("Classes:", CLASS_NAMES)

# Thống kê nhanh: multi-object & kích thước box (train)
areas, n_multi, n_img = [], 0, 0
for lp in (CLEAN_DATA_DIR / "train" / "labels").glob("*.txt"):
    lines = [ln for ln in lp.read_text().splitlines() if ln.strip()]
    n_img += 1
    if len(lines) >= 2:
        n_multi += 1
    for ln in lines:
        parts = ln.split()
        areas.append(float(parts[3]) * float(parts[4]))

import numpy as np
areas = np.array(areas)
print(f"Train images={n_img} | multi≥2={n_multi} ({100*n_multi/n_img:.1f}%)")
print("Box area_norm percentiles:", np.percentile(areas, [10, 50, 90]).round(3))
print(
    "→ Dataset đã có nhiều multi-object; vấn đề LIVE nghiêng về domain gap\n"
    "  (nền webcam + mặt sau + phân lớp mịn), không phải thiếu ảnh nhiều tờ trong Roboflow."
)

## 2. Recipe Model v2 (chốt cấu hình)

- **Backbone:** `yolov8s` (nhiều capacity hơn `n` → phân biệt mệnh giá gần nhau tốt hơn)  
- **Augment:** scale rộng, mosaic/mixup/copy_paste, erasing, HSV, xoay nhẹ  
- **Loss:** tăng `cls` để giảm nhầm class  
- **Train dài hơn:** 100 epoch, cosine LR, early stop patience 35  
- **Protocol:** cùng split Roboflow; chọn checkpoint theo **val**; so với baseline trên cùng val

In [ ]:
QUICK_RUN = False          # True = 5 epoch thử pipeline
RUN_TRAIN_HERE = False     # True nếu muốn train ngay trong notebook (cần GPU)
                           # False: chỉ lưu config + hướng dẫn chạy train_v2.py trên server

MODEL_NAME = "yolov8s.pt"
IMGSZ = 640
EPOCHS = 5 if QUICK_RUN else 100
BATCH = 24 if torch.cuda.is_available() else 4
SEED = 42
RUN_NAME = "vn_currency_yolov8s_v2"

V2_TRAIN_CFG = dict(
    data=str(CLEAN_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=8,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="AdamW",
    seed=SEED,
    deterministic=True,
    patience=35,
    save=True,
    plots=True,
    verbose=True,
    cos_lr=True,
    # augment
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=12.0,
    translate=0.15,
    scale=0.6,
    shear=2.0,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.25,
    erasing=0.25,
    close_mosaic=20,
    # loss
    box=7.5,
    cls=1.0,
    dfl=1.5,
)

cfg_path = V2_EXPORT / "v2_train_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump({"model": MODEL_NAME, **{k: str(v) if isinstance(v, Path) else v for k, v in V2_TRAIN_CFG.items()}}, f, indent=2, ensure_ascii=False)

print("Đã lưu config:", cfg_path)
print(pd.Series({k: V2_TRAIN_CFG[k] for k in [
    "epochs", "batch", "imgsz", "scale", "mosaic", "mixup", "copy_paste", "erasing", "cls", "degrees"
]}))

## 3. Train

### Cách A — Server (khuyến nghị, giống baseline)

```bash
# trên máy local: sync code + dataset lên server_02
rsync -avh --exclude '.venv' --exclude '.git' \
  ~/Desktop/Note/Hoc_Tap/Nhan_dang_mau/nhan_dang_tien_Viet_Nam/ \
  server_02@10.87.25.8:/home/server_02/Documents/dungtv/nhan_dang_tien_Viet_Nam/

# trên server
source ~/anaconda3/etc/profile.d/conda.sh
conda activate vn_currency
cd ~/Documents/dungtv/nhan_dang_tien_Viet_Nam
nohup python -u train_v2.py --epochs 100 --batch 24 > logs/train_v2.log 2>&1 &
tail -f logs/train_v2.log
```

### Cách B — Train trong notebook (bật `RUN_TRAIN_HERE = True`)

In [ ]:
RUN_DIR = None
BEST_PT = None

if RUN_TRAIN_HERE:
    model = YOLO(MODEL_NAME)
    model.train(**V2_TRAIN_CFG)
    RUN_DIR = Path(model.trainer.save_dir)
    BEST_PT = RUN_DIR / "weights" / "best.pt"
    print("BEST_PT:", BEST_PT)
else:
    # Thử tìm run đã train sẵn
    cand = sorted(RUNS_DIR.glob(f"{RUN_NAME}*"), key=lambda p: p.stat().st_mtime) if RUNS_DIR.exists() else []
    export_pt = V2_EXPORT / "yolov8s_v2_best.pt"
    if export_pt.exists():
        BEST_PT = export_pt
        print("Dùng weights đã export:", BEST_PT)
    elif cand and (cand[-1] / "weights" / "best.pt").exists():
        RUN_DIR = cand[-1]
        BEST_PT = RUN_DIR / "weights" / "best.pt"
        print("Dùng run có sẵn:", BEST_PT)
    else:
        print(
            "Chưa có checkpoint v2.\n"
            "→ Chạy train_v2.py trên server hoặc đặt RUN_TRAIN_HERE=True.\n"
            "Các cell so sánh bên dưới sẽ bỏ qua phần thiếu weights."
        )

## 4. Đánh giá v2 trên validation & so với baseline

In [ ]:
import numpy as np
def extract_metrics(metrics, class_names):
    box = metrics.box
    out = {
        "mAP50": float(box.map50),
        "mAP50-95": float(box.map),
        "precision": float(box.mp),
        "recall": float(box.mr),
    }
    if getattr(box, "ap50", None) is not None:
        ap50 = list(map(float, np.array(box.ap50).reshape(-1)))
        for i, name in enumerate(class_names):
            if i < len(ap50):
                out[f"AP50_{name}"] = ap50[i]
    return out


rows = []
if BASELINE_METRICS.exists():
    b = pd.read_csv(BASELINE_METRICS).iloc[0].to_dict()
    rows.append({
        "model": "A0 YOLOv8n baseline",
        "mAP50": b.get("val_mAP50"),
        "mAP50-95": b.get("val_mAP50-95"),
        "precision": b.get("val_precision"),
        "recall": b.get("val_recall"),
    })

v2_metrics = None
if BEST_PT is not None and Path(BEST_PT).exists():
    v2_model = YOLO(str(BEST_PT))
    v2_val = v2_model.val(data=str(CLEAN_YAML), split="val", imgsz=IMGSZ, device=DEVICE, plots=True, verbose=True)
    v2_metrics = extract_metrics(v2_val, CLASS_NAMES)
    rows.append({
        "model": "V2 YOLOv8s + strong aug",
        "mAP50": v2_metrics["mAP50"],
        "mAP50-95": v2_metrics["mAP50-95"],
        "precision": v2_metrics["precision"],
        "recall": v2_metrics["recall"],
    })

    # per-class AP cho các mệnh giá dễ nhầm
    hard = ["1000", "2000", "5000", "10000", "20000", "50000"]
    hard_df = pd.DataFrame([
        {"class": c, "AP50_v2": v2_metrics.get(f"AP50_{c}")} for c in hard
    ])
    display(hard_df)

    # copy export
    import shutil
    shutil.copy2(BEST_PT, V2_EXPORT / "yolov8s_v2_best.pt")
    pd.DataFrame([{
        "model": "YOLOv8s v2",
        "val_mAP50": v2_metrics["mAP50"],
        "val_mAP50-95": v2_metrics["mAP50-95"],
        "val_precision": v2_metrics["precision"],
        "val_recall": v2_metrics["recall"],
    }]).to_csv(V2_EXPORT / "v2_val_metrics.csv", index=False)

cmp = pd.DataFrame(rows)
display(cmp)
if len(cmp):
    cmp.to_csv(V2_EXPORT / "compare_baseline_vs_v2_val.csv", index=False)

if len(cmp) >= 2:
    melt = cmp.melt(id_vars="model", value_vars=["mAP50", "mAP50-95", "precision", "recall"],
                    var_name="metric", value_name="value")
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=melt, x="metric", y="value", hue="model", ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_title("Baseline vs V2 (validation)")
    plt.tight_layout()
    plt.show()

## 5. Sau khi có v2: cập nhật demo camera

```bash
python demo_camera.py \
  --weights notebooks/v2_exports/yolov8s_v2_best.pt \
  --mode live --conf 0.30
```

Gợi ý khi test lại LIVE:
- Giữ tờ tiền **xa hơn** (chiếm ~20–40% khung), nền vẫn rối — đây là test đúng mục tiêu v2  
- Thử **2 tờ** cạnh nhau  
- Thử **mặt sau 5000** — nếu vẫn fail: ưu tiên bổ sung data, không chỉ tăng epoch  
- Nếu còn box ảo trên mặt người: tăng `--conf 0.40` hoặc làm bước A3 (sweep threshold) ở notebook sau

## 6. Kết luận đề xuất cho nhóm

**Làm ngay (train v2):** YOLOv8s + augment mạnh + cls↑ — file `train_v2.py` / notebook này.  
**Song song (data):** 50–150 ảnh mới: nền rối + mặt sau 1000/2000/5000 + cặp nhầm — ROI cao nhất cho demo.  
**Sau v2:** notebook `05` = so sánh công bằng + sweep conf theo cost + phân tích lỗi trên ảnh tự chụp.

Không nên kỳ vọng chỉ tăng epoch YOLOv8n sẽ hết nhầm mặt sau / nền webcam.